# Controlled AHR + JEPA experiment

This controller carries the approved changes through training, three-seed comparisons, diagnostics, conditional latent-guidance experiments and a hash-verified local restore. The original completed project is preserved.

**Arms:** A is the frozen existing player; B uses consistent two-cycle AHR reasoning, broader tactical search and improved data/replay; C uses the same setup plus action-conditioned latent prediction. Both B/C receive 128 self-play games, 256 tactical warmup updates and 16 updates per recorded game, for each of three predeclared seeds. The plan compares C/B over 128 paired opening blocks per seed. Secondary comparisons use 32 paired blocks against A and eight against GNU Go level 1. Reference engines provide evaluation only. Their separately declared five-second whole-call allowance accounts for process/history replay, and is not an equal-compute claim.

**Inputs:** source-pinned notebooks, verified baseline, frozen family-separated curriculum, production recovery and real-pilot receipts. **Outputs:** all reports under `reports/ahr_jepa_v3/`, checkpoints under each named run, persistent trajectories, final local bundle. **Expected runtime:** multiple days; measured pilot timing is recorded before launch. **Default Run All:** imports/readiness only. `AHRV3_MODE=prepare` freezes the plan after final validation; `launch` starts detached tmux; `run` resumes the controller; `inspect` makes no training changes.

**Checkpoint/resume:** `checkpoints/pipeline-ahr-jepa-v3/` tracks completed stages and child process identity. Every trainer/arena owns its scientific recovery state and one-writer lock. Use the identical sources/configuration, fresh `ahr-go` kernel and `AHRV3_MODE=run`, Run All. Completed stages are skipped; invalid manifests/source drift and duplicate live children are rejected. Long runs use tmux session `ahr-go-jepa-v3`: attach `tmux attach -t ahr-go-jepa-v3`; detach **Ctrl-b, then d**. A VM reboot still requires resuming this notebook.

Latent guidance stays off unless heldout predictive, collapse, latency and controlled playing-strength gates pass; a passing gate permits a further comparison only. No model is called stronger from training loss alone, and no experimental model silently replaces the playable baseline. Local-only storage follows the user's instruction; off-VM backup is deferred.

In [ ]:
from pathlib import Path
import nbformat,os
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run within AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'v3-shared:export','exec'),globals())
v3_load_upstream(PROJECT,globals())
for name in ('08_predictive_ahr.ipynb','09_trajectory_learning.ipynb','11_controlled_evaluation.ipynb','13_local_handoff.ipynb','progress.ipynb'):
    v3_import_exports(PROJECT/V3_ROOT/name,globals())
_pipeline_namespace={'__name__':'__ahr_v3_pipeline__'}
for dependency in ('notebooks/00_instance_setup_and_recovery.ipynb','reports/09_13_training_pipeline.ipynb'):
    v3_import_exports(PROJECT/dependency,_pipeline_namespace)
# Keep the tested generic controller's implementation and isolated globals.
_v3_original_checkpoint_save=_pipeline_namespace['_save_checkpoint_unlocked']
def _v3_controller_checkpoint_save(*args,**kwargs):
    kwargs['resume_cell']='experiments/ahr_jepa_v3/protocol_and_run.ipynb: AHRV3_MODE=run; fresh kernel Run All'
    return _v3_original_checkpoint_save(*args,**kwargs)
_pipeline_namespace['_save_checkpoint_unlocked']=_v3_controller_checkpoint_save
v3_run_controller=_pipeline_namespace['run_training_pipeline']


In [ ]:
from dataclasses import asdict
import datetime as dt
import shutil,subprocess,sys,shlex

V3_PROTOCOL_PATH='config/ahr_jepa_v3.json'
V3_CONTROLLER='pipeline-ahr-jepa-v3'
V3_STAGE_NOTEBOOK='experiments/ahr_jepa_v3/run_stage.ipynb'
V3_PROTOCOL_NOTEBOOK='experiments/ahr_jepa_v3/protocol_and_run.ipynb'

def v3_all_experiment_sources(project):
    project=Path(project)
    paths=sorted((project/V3_ROOT).glob('*.ipynb'))
    paths+=sorted((project/'notebooks').glob('*.ipynb'))
    paths+=[project/'reports/09_13_training_pipeline.ipynb']
    return {str(p.relative_to(project)):v3_code_sha(p) for p in paths}

def v3_read_protocol(project):
    project=Path(project);config=v3_read_json(project/V3_PROTOCOL_PATH)
    expected=config['configuration_sha256']
    if v3_digest({k:v for k,v in config.items() if k!='configuration_sha256'})!=expected:raise RuntimeError('Protocol digest mismatch')
    if config['sources']!=v3_all_experiment_sources(project):raise RuntimeError('Frozen v3 experiment sources changed')
    if file_sha256(project/config['curriculum']['path'])!=config['curriculum']['sha256']:raise RuntimeError('Frozen curriculum changed')
    v3_verify_original_sources(project)
    return config

def v3_training_config(protocol,arm,index):
    if arm not in ('B','C') or not 1<=index<=3:raise ValueError('Declared arm/seed required')
    values=copy.deepcopy(protocol['training'])
    values.update(seed=protocol['training_seeds'][index-1],use_jepa=arm=='C',
                  initial_run=protocol['baseline_run'],curriculum_path=protocol['curriculum']['path'])
    return V3TrainConfig(**values)

def v3_stage_plan(protocol):
    plan=[{'kind':'baseline_diagnostics','index':0,'arm':'A'}]
    for index in range(1,4):
        for arm in ('B','C'):plan.append({'kind':'train','index':index,'arm':arm})
        for comparison in ('C_vs_B','B_vs_A','C_vs_A','B_vs_GNU','C_vs_GNU'):
            plan.append({'kind':'compare','index':index,'comparison':comparison})
        for arm in ('B','C'):plan.append({'kind':'diagnostics','index':index,'arm':arm})
    plan.append({'kind':'family','index':0})
    for index in range(1,4):plan.append({'kind':'guided_comparison','index':index,'arm':'C'})
    plan.extend([{'kind':'bundle','index':0},{'kind':'report','index':0}])
    for ordinal,stage in enumerate(plan):
        suffix=stage.get('comparison',stage.get('arm',''))
        stage['name']=f"{stage['kind']}-{suffix}-s{stage['index']:02d}".strip('-')
        stage['ordinal']=ordinal
    return plan

def v3_controller_stages(protocol):
    return [{'name':s['name'],'notebook':V3_STAGE_NOTEBOOK,
             'env':{'AHRV3_STAGE_INDEX':str(s['ordinal'])},'require':'process_success'} for s in v3_stage_plan(protocol)]

def v3_prepare_protocol(project):
    project=Path(project);path=project/V3_PROTOCOL_PATH
    if path.exists():return v3_read_protocol(project)
    validation=v3_read_json(v3_report_path(project,'delivery_validation.json'))
    if validation.get('status')!='PASS' or validation['sources']!=v3_all_experiment_sources(project):raise RuntimeError('Final fresh-kernel delivery validation is required before freeze')
    if not validation.get('production_recovery_passed') or not validation.get('real_pilot_passed'):raise RuntimeError('Production recovery and real pilot are required')
    if torch.cuda.device_count()<2:raise RuntimeError('Protocol requires the validated two-device topology')
    if shutil.disk_usage(project).free<30*1024**3:raise RuntimeError('Insufficient workspace capacity for persistent trajectories and backups')
    baseline,metadata=load_go_ahr(project,V3_BASELINE,device='cpu',selection='latest');del baseline
    if metadata['checkpoint_sha256']!=V3_BASELINE_CHECKPOINT:raise RuntimeError('Frozen baseline checkpoint changed')
    curriculum_data,curriculum=v3_read_curriculum(project)
    if not all(curriculum['splits'].get(s,0)>0 for s in ('train','development','lockbox')):raise RuntimeError('Missing separated curriculum split')
    seeds=[2026092701,2026092702,2026092703]
    config={'schema':V3_SCHEMA,'controller':V3_CONTROLLER,'training_seeds':seeds,
        'baseline_run':V3_BASELINE,'baseline_checkpoint_sha256':V3_BASELINE_CHECKPOINT,
        'curriculum':curriculum,'family':v3_evaluation_family(seeds,blocks=128,opening_seed=2026092711),
        'training':{'games':128,'warmup_updates':256,'updates_per_game':16,'batch_size':32,
                    'device':'cuda:0','actor_device':'cuda:1','replay_capacity_games':4096,
                    'time_budget_s':1.0,'max_plies':256,
                    'search':{'max_depth':4,'max_nodes':192,'branch_width':8,'queen_cycles':2}},
        'evaluation':{'device':'cuda:1','max_cases':64,'max_sequences':64,'inference_budget_s':1.0,
                      'external_blocks':8,'external_level':1,'reference_budget_s':5.0,'reference_command_timeout_s':3.0},
        'fairness':'Three predeclared seeds, identical starting weights per seed, shared curriculum, equal games/updates/sampling plan and B/C inference budget. Actual training wall times are reported; no exact FLOP equality assertion.',
        'latent_inference':'Disabled by default. Separate gated C_guided versus C comparison only after heldout predictive/collapse/latency and three-seed C versus B strength evidence.',
        'selection':'All seeds reported; no best-seed selection. Current playable baseline remains until a separate complete promotion decision.',
        'off_instance_backup':'DEFERRED_BY_USER_PENDING_HUGGING_FACE_DESTINATION',
        'sources':v3_all_experiment_sources(project),'validation_sha256':file_sha256(v3_report_path(project,'delivery_validation.json')),
        'created_at_utc':dt.datetime.now(dt.timezone.utc).isoformat()}
    for index in range(1,4):
        for arm in ('B','C'):v3_training_config(config,arm,index)
    config['configuration_sha256']=v3_digest(config)
    atomic_json(path,config)
    return config

def v3_comparison_config(protocol,index,comparison):
    seed=protocol['training_seeds'][index-1]
    arm,opponent=comparison.split('_vs_')
    candidate=f'ahr-v3-{arm}-s{index:02d}'
    other=protocol['baseline_run'] if opponent=='A' else (None if opponent=='GNU' else f'ahr-v3-{opponent}-s{index:02d}')
    family=protocol['family'];primary=comparison=='C_vs_B'
    blocks=family['blocks'] if primary else protocol['evaluation']['external_blocks'] if opponent=='GNU' else family['secondary_blocks']
    settings={'candidate_run':candidate,'candidate_arm':arm,'opponent_run':other,'opponent_arm':opponent,
        'training_seed':seed,'blocks':blocks,'device':protocol['evaluation']['device'],'max_plies':256,
        'inference_budget_s':1.,'cycles':2,'alpha':family['alpha_per_seed'] if primary else .05,
        'reference_level':protocol['evaluation']['external_level'],'reference_budget_s':protocol['evaluation']['reference_budget_s'],
        'reference_command_timeout_s':protocol['evaluation']['reference_command_timeout_s']}
    if primary:settings['opening_seeds']=family['opening_seeds'][str(seed)]
    else:settings['opening_seed']=seed+({'B_vs_A':11000,'C_vs_A':11000,'B_vs_GNU':22000,'C_vs_GNU':22000}[comparison])
    return settings

def v3_dispatch_stage(project,ordinal):
    project=Path(project);protocol=v3_read_protocol(project);plan=v3_stage_plan(protocol)
    if type(ordinal)is not int or not 0<=ordinal<len(plan):raise ValueError('Declared stage ordinal required')
    stage=plan[ordinal];index=stage['index'];kind=stage['kind'];directory=project/'reports/ahr_jepa_v3'
    invoked_at=dt.datetime.now(dt.timezone.utc).isoformat();invocation_start=time.perf_counter()
    with checkpoint_lock(project/'checkpoints'/(V3_CONTROLLER+'-stage')):
        if kind=='train':
            run=f"ahr-v3-{stage['arm']}-s{index:02d}"
            result=v3_run_training(project,run,v3_training_config(protocol,stage['arm'],index))
            report=v3_read_json(v3_report_path(project,run+'_training.json'))
            if report.get('status',report.get('phase','')).lower()!='complete':raise RuntimeError('Training stage did not complete')
            result=report
        elif kind=='compare':
            run=f"arena-v3-{stage['comparison']}-s{index:02d}"
            snapshot,result=v3_run_evaluation(project,run,v3_comparison_config(protocol,index,stage['comparison']))
            if not snapshot['complete']:raise RuntimeError('Declared arena has unfinished jobs')
        elif kind in ('baseline_diagnostics','diagnostics'):
            arm=stage['arm'];run=protocol['baseline_run'] if arm=='A' else f'ahr-v3-{arm}-s{index:02d}'
            arenas=[] if arm=='A' else [f'arena-v3-C_vs_B-s{index:02d}',f'arena-v3-{arm}_vs_A-s{index:02d}']
            result=v3_evaluate_diagnostics(project,run,{'arm':arm,'curriculum_descriptor':protocol['curriculum'],
                'max_cases':protocol['evaluation']['max_cases'],'max_sequences':protocol['evaluation']['max_sequences'],
                'arena_run_ids':arenas,'device':protocol['evaluation']['device']})
        elif kind=='family':
            reports=[v3_read_json(directory/(f'arena-v3-{comparison}-s{index:02d}-results.json'))
                for index in range(1,4) for comparison in ('C_vs_B','B_vs_A','C_vs_A','B_vs_GNU','C_vs_GNU')]
            result=v3_family_summary(project,protocol['family'],reports)
        elif kind=='guided_comparison':
            run=f'ahr-v3-C-s{index:02d}'
            diagnostics=v3_read_json(directory/('diagnostics-'+run+'.json'))
            family=v3_read_json(directory/'family_summary.json')
            gate=v3_latent_guidance_gate(project,run,diagnostics,family)
            if gate['status']!='PASS':
                result={'status':'SKIPPED_BY_DECLARED_GATE','run_id':run,'gate':gate,'training_only_predictor':True}
            else:
                request={'candidate_run':run,'candidate_arm':'C_guided','opponent_run':run,'opponent_arm':'C',
                    'training_seed':protocol['training_seeds'][index-1],'blocks':128,'opening_seed':2027092700+index,
                    'device':protocol['evaluation']['device'],'max_plies':256,'inference_budget_s':1.,'cycles':2,
                    'alpha':.05/3,'latent_gate':gate}
                snapshot,result=v3_run_evaluation(project,f'arena-v3-guided-s{index:02d}',request)
                if not snapshot['complete']:raise RuntimeError('Guided comparison unfinished')
        elif kind=='bundle':
            runs=[f'ahr-v3-{arm}-s{index:02d}' for index in range(1,4) for arm in ('B','C')]
            result=v3_export_experiment(project,runs)
        elif kind=='report':
            family=v3_read_json(directory/'family_summary.json');bundle=v3_read_json(directory/'local_bundle_final.json')
            if bundle['status']!='VERIFIED_LOCAL':raise RuntimeError('Local restore did not pass')
            result={'schema':V3_SCHEMA,'status':'COMPLETE_LOCAL_EXPERIMENT','family':family,
                'selected_playable_run':protocol['baseline_run'],'model_selection':'No automatic deployment from experimental evidence',
                'local_bundle':bundle,'off_instance_backup':protocol['off_instance_backup'],
                'completed_at_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
                'reports':[str(p.relative_to(project)) for p in sorted(directory.glob('*.json'))]}
            atomic_json(directory/'completion.json',result)
        else:raise AssertionError(kind)
        atomic_json(directory/('stage-'+stage['name']+'.json'),{'schema':V3_SCHEMA,'stage':stage,'result':result,
            'configuration_sha256':protocol['configuration_sha256'],'started_this_invocation_at_utc':invoked_at,
            'elapsed_this_invocation_s':time.perf_counter()-invocation_start,'finished_at_utc':dt.datetime.now(dt.timezone.utc).isoformat()})
        v3_refresh_progress(project)
        return result

def v3_launch_detached(project):
    project=Path(project);protocol=v3_read_protocol(project);session='ahr-go-jepa-v3'
    with checkpoint_lock(project/'checkpoints'/(V3_CONTROLLER+'-launch')):
        if subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0:
            return {'status':'EXISTING_SESSION','session':session,'action':'Inspect existing session; no second writer launched'}
        command=['env','AHRV3_MODE=run',sys.executable,'-u','-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
                 '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(project/V3_PROTOCOL_NOTEBOOK)]
        log=project/'reports/ahr_jepa_v3/controller.log'
        subprocess.run(['tmux','new-session','-d','-s',session,'-n','pipeline','-c',str(project),
            'exec '+shlex.join(command)+' >> '+shlex.quote(str(log))+' 2>&1'],check=True)
        subprocess.run(['tmux','set-window-option','-t',session+':pipeline','remain-on-exit','on'],check=True)
        monitor=['env','AHRV3_MONITOR=1',sys.executable,'-u','-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
                 '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(project/V3_ROOT/'monitor.ipynb')]
        subprocess.run(['tmux','new-window','-d','-t',session,'-n','progress','-c',str(project),
            'exec '+shlex.join(monitor)+' >> '+shlex.quote(str(project/'reports/ahr_jepa_v3/monitor.log'))+' 2>&1'],check=True)
        receipt={'status':'LAUNCHED','session':session,'configuration_sha256':protocol['configuration_sha256'],
                 'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d','controller_log':str(log.relative_to(project)),
                 'resume':'AHRV3_MODE=run; fresh ahr-go kernel Run All in '+V3_PROTOCOL_NOTEBOOK}
        atomic_json(v3_report_path(project,'launch.json'),receipt)
        return receipt


In [ ]:
mode=os.environ.get('AHRV3_MODE','inspect')
if mode=='prepare':
    protocol=v3_prepare_protocol(PROJECT)
    print(json.dumps({'status':'FROZEN','configuration_sha256':protocol['configuration_sha256'],'stages':len(v3_stage_plan(protocol))},indent=2))
elif mode=='launch':print(json.dumps(v3_launch_detached(PROJECT),indent=2))
elif mode=='run':
    protocol=v3_read_protocol(PROJECT)
    result=v3_run_controller(PROJECT,V3_CONTROLLER,stages=v3_controller_stages(protocol))
    v3_refresh_progress(PROJECT)
    print(json.dumps({'status':result['status'],'completed_stages':result['stage_index']},indent=2))
elif mode=='inspect':
    print(json.dumps({'status':'INSPECT_ONLY','original_sources':v3_verify_original_sources(PROJECT),
        'protocol_frozen':(PROJECT/V3_PROTOCOL_PATH).exists(),'training_started':False},indent=2))
else:raise ValueError('Mode must be inspect/prepare/launch/run')